# Causalyst Nifty 500 Temporal GNN Colab Notebook

Scales the validated `gnn_prototype.py` architecture to Nifty 500 on free Google Colab:

- per-node GRU encoder over rolling return windows
- two base `torch_geometric.nn.GCNConv` graph-propagation layers
- prediction head over `[self_embedding, graph_embedding]`

Phase A constructs and caches the graph to Google Drive. Phase B trains the temporal GNN and two baselines with resumable Drive checkpoints. The primary 500-stock graph uses parallel pairwise Granger plus Benjamini-Hochberg FDR, not Bonferroni. PCMCI is not run on the full 500-stock joint set unless a 50-stock timing gate is run first.

Dataset source: [Ratnesh-bhosale/NIFTY500_dataset](https://github.com/Ratnesh-bhosale/NIFTY500_dataset).

In [ ]:
# PHASE 0 - Mount Google Drive first. All durable outputs go under DRIVE_ROOT.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    IN_COLAB = False
    print(f"Not running inside Colab or Drive mount failed: {exc}")
    print("For a real run, execute this notebook in Google Colab and allow Drive access.")

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/causalyst_nifty500_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = DRIVE_ROOT / 'data'
GRAPH_DIR = DRIVE_ROOT / 'graphs'
PAIR_RESULTS_DIR = GRAPH_DIR / 'granger_pair_results'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
LOG_DIR = DRIVE_ROOT / 'logs'

for path in [DATA_ROOT, GRAPH_DIR, PAIR_RESULTS_DIR, CHECKPOINT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Durable Drive root: {DRIVE_ROOT}")

In [ ]:
# Install runtime dependencies. Torch is usually preinstalled on Colab; torch-geometric may not be.
import importlib.util
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'statsmodels', 'networkx', 'scikit-learn', 'tqdm', 'requests'
])

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'])

print('Dependencies ready.')

In [ ]:
# Imports and fixed experiment configuration.
import contextlib
import gc
import io
import json
import math
import multiprocessing as mp
import os
import random
import shutil
import subprocess
import sys
import time
import warnings
from datetime import datetime
from itertools import permutations

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import grangercausalitytests
from torch.utils.data import DataLoader, Dataset
from torch_geometric.nn import GCNConv
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# User-approved chronological split.
TRAIN_END = pd.Timestamp('2019-12-31')
VAL_START = pd.Timestamp('2020-01-01')
VAL_END = pd.Timestamp('2020-12-31')
TEST_START = pd.Timestamp('2021-01-01')
TEST_END = pd.Timestamp('2021-12-31')

WINDOW = 20
MAX_LAG = 2
FDR_ALPHA = 0.05
MEASURED_SECONDS_PER_PAIR = 0.00895
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)
SAVE_EVERY_RESULTS = 5_000
MAX_STOCKS = 500
MIN_COVERAGE = 0.85

BATCH_SIZE = 8
EPOCHS = 20
CHECKPOINT_EVERY_EPOCHS = 2
LEARNING_RATE = 1e-3
HIDDEN = 16

CAUSAL_GRAPH_PATH = GRAPH_DIR / 'nifty500_granger_fdr_graph.pt'
CAUSAL_EDGE_CSV = GRAPH_DIR / 'nifty500_granger_fdr_edges.csv'
CORR_GRAPH_PATH = GRAPH_DIR / 'nifty500_correlation_graph.pt'
CLUSTER_REPORT_PATH = GRAPH_DIR / 'nifty500_louvain_clusters.json'
TRAINING_SUMMARY_PATH = LOG_DIR / 'training_summary.json'
EVENT_LOG_PATH = LOG_DIR / 'events.jsonl'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | CPU workers for Granger: {N_WORKERS}")
print(f"Split: train <= {TRAIN_END.date()}, val {VAL_START.date()}..{VAL_END.date()}, test {TEST_START.date()}..{TEST_END.date()}")


def log_event(event, **payload):
    row = {'ts': datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'event': event, **payload}
    with EVENT_LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, default=str) + '\n')
    print(json.dumps(row, default=str))


def format_seconds(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"

## Phase A1 - Download And Load Nifty 500 CSVs

The clone lives on Drive, so Colab disconnects do not require re-downloading. The loader recursively finds CSVs with `Date` and `Close`, aligns closes, filters to sufficient coverage, and computes daily log returns.

In [ ]:
DATASET_REPO_URL = 'https://github.com/Ratnesh-bhosale/NIFTY500_dataset.git'
DATASET_DIR = DATA_ROOT / 'NIFTY500_dataset'

if not DATASET_DIR.exists():
    log_event('dataset_clone_start', repo=DATASET_REPO_URL, target=str(DATASET_DIR))
    subprocess.check_call(['git', 'clone', '--depth=1', DATASET_REPO_URL, str(DATASET_DIR)])
    log_event('dataset_clone_complete', target=str(DATASET_DIR))
else:
    print(f"Dataset already present on Drive: {DATASET_DIR}")


def normalize_ticker_from_path(path):
    stem = path.stem.upper().strip()
    parts = stem.split('_')
    if len(parts) > 1 and parts[0].isdigit():
        stem = '_'.join(parts[1:])
    return stem.replace('.NS', '').replace('-EQ', '').replace(' ', '')


def find_price_csvs(root):
    candidates = []
    for path in sorted(root.rglob('*.csv')):
        try:
            header = pd.read_csv(path, nrows=1)
        except Exception:
            continue
        lower = {c.lower(): c for c in header.columns}
        if 'date' in lower and 'close' in lower:
            candidates.append(path)
    return candidates


def load_nifty500_prices(root, max_stocks=MAX_STOCKS):
    csv_paths = find_price_csvs(root)
    print(f"Found {len(csv_paths)} price-like CSV files.")
    series = {}
    bad_files = []
    for path in tqdm(csv_paths, desc='Loading close prices'):
        ticker = normalize_ticker_from_path(path)
        if ticker in series:
            continue
        try:
            df = pd.read_csv(path, parse_dates=['Date']).sort_values('Date').set_index('Date')
            close = pd.to_numeric(df['Close'], errors='coerce').dropna()
            if close.empty:
                continue
            series[ticker] = close
        except Exception as exc:
            bad_files.append((str(path), str(exc)))
    if not series:
        raise RuntimeError('No usable price CSVs found. Inspect DATASET_DIR and CSV columns.')

    prices = pd.DataFrame(series).sort_index()
    prices = prices.loc[:TEST_END]
    coverage = prices.notna().mean()
    keep = coverage[coverage >= MIN_COVERAGE].sort_index().index.tolist()
    prices = prices[keep].ffill().bfill().dropna(axis=1)
    if max_stocks is not None and prices.shape[1] > max_stocks:
        prices = prices.iloc[:, :max_stocks]
    returns = np.log(prices / prices.shift(1)).dropna()
    returns = returns.loc[:TEST_END]

    print(f"Loaded prices: {prices.shape[0]} dates x {prices.shape[1]} stocks")
    print(f"Returns: {returns.shape[0]} dates x {returns.shape[1]} stocks")
    print(f"Date range: {returns.index.min().date()} to {returns.index.max().date()}")
    if bad_files:
        print(f"Skipped {len(bad_files)} malformed CSVs. First skipped file: {bad_files[0]}")
    if returns.shape[1] < 450:
        print('WARNING: fewer than 450 stocks survived loading/coverage filters. Check dataset contents before treating this as full Nifty 500.')
    return prices, returns


prices_df, returns_df = load_nifty500_prices(DATASET_DIR)
tickers = list(returns_df.columns)
log_event('data_loaded', n_days=len(returns_df), n_stocks=len(tickers), start=str(returns_df.index.min().date()), end=str(returns_df.index.max().date()))

## Phase A2 - Parallel Granger + Benjamini-Hochberg FDR

Measured timing is fixed input: **8.95 ms per ordered pair**. At 500 stocks, that is about 249,500 ordered pairs and about 37 minutes single-threaded. This cell uses `multiprocessing.Pool` with `max(1, os.cpu_count() - 1)` workers and saves raw pair-result shards to Drive every 5,000 pairs. If the session disconnects mid-run, rerun the notebook and only missing pairs are scheduled.

In [ ]:
_GRANGER_RETURNS = None


def _init_granger_worker(returns_payload):
    global _GRANGER_RETURNS
    _GRANGER_RETURNS = returns_payload


def _granger_pair(pair):
    a, b = pair
    try:
        sub = _GRANGER_RETURNS[[b, a]].dropna()
        best = None
        for lag in range(1, MAX_LAG + 1):
            if len(sub) <= lag * 4:
                continue
            try:
                with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):
                    warnings.simplefilter('ignore')
                    result = grangercausalitytests(sub, maxlag=[lag], verbose=False)
                f_test = result[lag][0]['ssr_ftest']
                f_stat, p_value = float(f_test[0]), float(f_test[1])
            except Exception:
                continue
            if best is None or p_value < best['p_value']:
                best = {'lag': lag, 'p_value': p_value, 'f_stat': f_stat}
        if best is None:
            return {'source': a, 'target': b, 'lag': np.nan, 'p_value': np.nan, 'f_stat': np.nan, 'ok': False, 'error': 'no_valid_lag'}
        return {'source': a, 'target': b, **best, 'ok': True, 'error': ''}
    except Exception as exc:
        return {'source': a, 'target': b, 'lag': np.nan, 'p_value': np.nan, 'f_stat': np.nan, 'ok': False, 'error': str(exc)[:180]}


def load_completed_pair_results():
    frames = []
    for path in sorted(PAIR_RESULTS_DIR.glob('part_*.csv')):
        try:
            frames.append(pd.read_csv(path))
        except Exception as exc:
            print(f"Could not read {path}: {exc}")
    if not frames:
        return pd.DataFrame(columns=['source', 'target', 'lag', 'p_value', 'f_stat', 'ok', 'error'])
    return pd.concat(frames, ignore_index=True).drop_duplicates(['source', 'target'], keep='last')


def next_part_path():
    existing = sorted(PAIR_RESULTS_DIR.glob('part_*.csv'))
    if not existing:
        return PAIR_RESULTS_DIR / 'part_000000.csv'
    last = max(int(p.stem.split('_')[-1]) for p in existing)
    return PAIR_RESULTS_DIR / f'part_{last + 1:06d}.csv'


def save_pair_buffer(buffer):
    if not buffer:
        return
    out = next_part_path()
    pd.DataFrame(buffer).to_csv(out, index=False)
    print(f"Saved {len(buffer)} pair results -> {out}")


def run_parallel_granger_fdr(returns_df, tickers):
    if CAUSAL_GRAPH_PATH.exists() and CAUSAL_EDGE_CSV.exists():
        print(f"Causal graph cache exists: {CAUSAL_GRAPH_PATH}")
        return torch.load(CAUSAL_GRAPH_PATH, map_location='cpu')

    all_pairs = list(permutations(tickers, 2))
    n_pairs = len(all_pairs)
    single_thread_est = MEASURED_SECONDS_PER_PAIR * n_pairs
    parallel_est = single_thread_est / max(N_WORKERS, 1)
    print(f"Ordered pairs: {n_pairs:,}")
    print(f"Measured estimate: {MEASURED_SECONDS_PER_PAIR*1000:.2f} ms/pair")
    print(f"Single-thread estimate: {format_seconds(single_thread_est)}")
    print(f"Parallel estimate with {N_WORKERS} workers: {format_seconds(parallel_est)}")

    completed_df = load_completed_pair_results()
    done = set(zip(completed_df['source'], completed_df['target'])) if len(completed_df) else set()
    remaining_pairs = [pair for pair in all_pairs if pair not in done]
    print(f"Completed from Drive shards: {len(done):,}; remaining: {len(remaining_pairs):,}")

    buffer = []
    start = time.time()
    pool = None
    try:
        if remaining_pairs:
            if N_WORKERS == 1:
                _init_granger_worker(returns_df)
                iterator = map(_granger_pair, remaining_pairs)
            else:
                pool = mp.Pool(processes=N_WORKERS, initializer=_init_granger_worker, initargs=(returns_df,))
                iterator = pool.imap_unordered(_granger_pair, remaining_pairs, chunksize=25)
            with tqdm(total=len(remaining_pairs), desc='Parallel Granger pairs') as pbar:
                for result in iterator:
                    buffer.append(result)
                    pbar.update(1)
                    if pbar.n % 500 == 0:
                        elapsed = time.time() - start
                        rate = pbar.n / elapsed if elapsed > 0 else 0
                        eta = (len(remaining_pairs) - pbar.n) / rate if rate > 0 else 0
                        pbar.set_postfix(rate=f"{rate:.1f}/s", eta=format_seconds(eta))
                    if len(buffer) >= SAVE_EVERY_RESULTS:
                        save_pair_buffer(buffer)
                        buffer.clear()
            if pool is not None:
                pool.close()
                pool.join()
        save_pair_buffer(buffer)
    except BaseException:
        save_pair_buffer(buffer)
        if pool is not None:
            pool.terminate()
            pool.join()
        log_event('granger_interrupted_pair_results_saved')
        raise

    pair_df = load_completed_pair_results()
    ok_df = pair_df[pair_df['ok'].astype(str).isin(['True', 'true', '1'])].copy().dropna(subset=['p_value'])
    reject, p_adj, _, _ = multipletests(ok_df['p_value'].values, alpha=FDR_ALPHA, method='fdr_bh')
    ok_df['p_adj_fdr_bh'] = p_adj
    edge_df = ok_df.loc[reject].copy().sort_values('p_adj_fdr_bh')
    edge_df.to_csv(CAUSAL_EDGE_CSV, index=False)

    idx = {ticker: i for i, ticker in enumerate(tickers)}
    src = [idx[a] for a in edge_df['source']]
    tgt = [idx[b] for b in edge_df['target']]
    weights = [-math.log10(max(float(p), 1e-12)) for p in edge_df['p_value']]
    edge_index = torch.tensor([src, tgt], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    payload = {
        'ticker_list': tickers,
        'edge_index': edge_index,
        'edge_weight': edge_weight,
        'edge_records': edge_df.to_dict('records'),
        'method': 'pairwise_granger_fdr_bh',
        'max_lag': MAX_LAG,
        'fdr_alpha': FDR_ALPHA,
        'n_pair_tests': int(len(ok_df)),
        'n_edges': int(len(edge_df)),
        'created_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    }
    torch.save(payload, CAUSAL_GRAPH_PATH)
    log_event('causal_graph_saved', path=str(CAUSAL_GRAPH_PATH), n_edges=len(edge_df), n_pair_tests=len(ok_df))
    return payload


causal_graph = run_parallel_granger_fdr(returns_df, tickers)
print(f"Causal graph edges after BH-FDR: {causal_graph['n_edges']:,}")
print(f"Saved immediately to Drive: {CAUSAL_GRAPH_PATH}")

## Phase A3 - Louvain Clustering And Modularity

Uses the same logic as `graph_clustering.py`: symmetrize directed causal edges into an undirected weighted graph with edge weight `-log10(p_value)`, run NetworkX Louvain community detection, and report modularity honestly.

In [ ]:
def build_undirected_weighted_graph_from_edges(edge_df):
    g = nx.Graph()
    for _, row in edge_df.iterrows():
        src, tgt = row['source'], row['target']
        w = -math.log10(max(float(row['p_value']), 1e-12))
        if g.has_edge(src, tgt):
            g[src][tgt]['weight'] += w
        else:
            g.add_edge(src, tgt, weight=w)
    for ticker in tickers:
        g.add_node(ticker)
    return g


edge_payload = causal_graph.get('edge_records', causal_graph.get('edge_df', []))
edge_df = edge_payload if isinstance(edge_payload, pd.DataFrame) else pd.DataFrame(edge_payload)
g_cluster = build_undirected_weighted_graph_from_edges(edge_df)
communities = nx.community.louvain_communities(g_cluster, weight='weight', seed=SEED) if g_cluster.number_of_edges() else [{n} for n in g_cluster.nodes]
modularity = nx.community.modularity(g_cluster, communities, weight='weight') if g_cluster.number_of_edges() else 0.0
cluster_report = {
    'method': 'networkx_louvain_on_graph_clustering_style_weighted_symmetrization',
    'n_nodes': g_cluster.number_of_nodes(),
    'n_edges_undirected': g_cluster.number_of_edges(),
    'n_clusters': len(communities),
    'modularity': float(modularity),
    'clusters': [sorted(list(c)) for c in sorted(communities, key=lambda x: (-len(x), sorted(x)))],
}
CLUSTER_REPORT_PATH.write_text(json.dumps(cluster_report, indent=2), encoding='utf-8')
print(f"Louvain clusters: {cluster_report['n_clusters']} | modularity: {modularity:.4f}")
print(f"Largest cluster sizes: {[len(c) for c in cluster_report['clusters'][:10]]}")
print(f"Saved cluster report: {CLUSTER_REPORT_PATH}")
log_event('louvain_complete', n_clusters=len(communities), modularity=float(modularity))

## Optional PCMCI Scale Gate

PCMCI is **not** used as the full 500-stock primary method here. If you want to explore PCMCI, first set `RUN_PCMCI_50_TIMING = True` and time around 50 stocks. Use that number before deciding on sector-restricted PCMCI refinement.

In [ ]:
RUN_PCMCI_50_TIMING = False

if RUN_PCMCI_50_TIMING:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tigramite'])
        from tigramite import data_processing as pp
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite.pcmci import PCMCI

        sample_returns = returns_df.iloc[:, :50]
        dataframe = pp.DataFrame(sample_returns.values, var_names=list(sample_returns.columns))
        pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=0)
        t0 = time.time()
        _ = pcmci.run_pcmci(tau_max=MAX_LAG, pc_alpha=0.05)
        elapsed = time.time() - t0
        print(f"PCMCI timing gate: 50 stocks x {len(sample_returns)} days took {format_seconds(elapsed)}")
        print('Decision rule: use this only for within-sector refinement unless this timing is comfortably cheap.')
        log_event('pcmci_50_timing_complete', seconds=elapsed, n_stocks=50)
    except Exception as exc:
        print(f"PCMCI timing failed or unavailable: {exc}")
        log_event('pcmci_50_timing_failed', error=str(exc))
else:
    print('PCMCI 50-stock timing skipped. Full 500-stock PCMCI is intentionally not run by this notebook.')

## Phase B1 - Build Chronological Train/Val/Test Windows

Labels are next-day direction (`return[t+1] > 0`) per stock.

In [ ]:
class StockWindowDataset(Dataset):
    def __init__(self, returns_df, label_start=None, label_end=None, window=WINDOW):
        self.returns = torch.tensor(returns_df.values, dtype=torch.float32)
        self.dates = pd.DatetimeIndex(returns_df.index)
        self.window = window
        indices = []
        for t in range(window, len(self.dates) - 1):
            label_date = self.dates[t + 1]
            if label_start is not None and label_date < pd.Timestamp(label_start):
                continue
            if label_end is not None and label_date > pd.Timestamp(label_end):
                continue
            indices.append(t)
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        t = self.indices[item]
        x = self.returns[t - self.window:t, :].T
        y = (self.returns[t + 1, :] > 0).float()
        return x, y


train_ds = StockWindowDataset(returns_df, label_start=None, label_end=TRAIN_END)
val_ds = StockWindowDataset(returns_df, label_start=VAL_START, label_end=VAL_END)
test_ds = StockWindowDataset(returns_df, label_start=TEST_START, label_end=TEST_END)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train windows: {len(train_ds):,} | Val windows: {len(val_ds):,} | Test windows: {len(test_ds):,}")
print(f"Each sample: x=(n_stocks={len(tickers)}, window={WINDOW}), y=(n_stocks={len(tickers)})")
log_event('windows_built', train=len(train_ds), val=len(val_ds), test=len(test_ds), n_stocks=len(tickers))

## Phase B2 - CausalTemporalGNN Architecture

Preserves `gnn_prototype.py`: per-node GRU encoder, two base `GCNConv` layers, concatenate self and graph embeddings, prediction head. The forward path supports mini-batches by repeating the same graph with node-index offsets.

In [ ]:
class CausalTemporalGNN(nn.Module):
    def __init__(self, hidden=HIDDEN, use_gcn=True):
        super().__init__()
        self.hidden = hidden
        self.use_gcn = use_gcn
        self.node_gru = nn.GRU(input_size=1, hidden_size=hidden, batch_first=True)
        self.gcn1 = GCNConv(hidden, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self._batched_graph_cache = {}

    def _batch_graph(self, edge_index, edge_weight, batch_size, n_nodes, device):
        key = (batch_size, n_nodes, edge_index.shape[1], device.type, str(device))
        if key in self._batched_graph_cache:
            return self._batched_graph_cache[key]
        edge_index = edge_index.to(device)
        edge_weight = edge_weight.to(device)
        if edge_index.numel() == 0:
            out_index = edge_index
            out_weight = edge_weight
        else:
            offsets = torch.arange(batch_size, device=device, dtype=torch.long).repeat_interleave(edge_index.shape[1]) * n_nodes
            out_index = edge_index.repeat(1, batch_size) + offsets.unsqueeze(0)
            out_weight = edge_weight.repeat(batch_size)
        self._batched_graph_cache[key] = (out_index, out_weight)
        return out_index, out_weight

    def forward(self, x, edge_index, edge_weight):
        squeeze = False
        if x.dim() == 2:
            x = x.unsqueeze(0)
            squeeze = True
        batch_size, n_nodes, window = x.shape
        x_seq = x.reshape(batch_size * n_nodes, window, 1)
        _, h_n = self.node_gru(x_seq)
        self_embed = h_n.squeeze(0)

        if self.use_gcn:
            batched_edge_index, batched_edge_weight = self._batch_graph(edge_index, edge_weight, batch_size, n_nodes, x.device)
            g1 = torch.relu(self.gcn1(self_embed, batched_edge_index, batched_edge_weight))
            g2 = self.gcn2(g1, batched_edge_index, batched_edge_weight)
        else:
            g2 = torch.zeros_like(self_embed)

        combined = torch.cat([self_embed, g2], dim=-1)
        logits = self.head(combined).squeeze(-1).view(batch_size, n_nodes)
        return logits.squeeze(0) if squeeze else logits


def empty_graph(_n_nodes):
    return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)


print(CausalTemporalGNN())
print(f"Parameters: {sum(p.numel() for p in CausalTemporalGNN().parameters()):,}")

## Phase B3 - Correlation-Graph Baseline

The correlation graph uses top-`K` absolute train-period correlations, with `K` matched to causal graph edge count. The GRU-only baseline reuses the same model class with `use_gcn=False`.

In [ ]:
def load_graph_tensors(graph_payload):
    return graph_payload['edge_index'].long(), graph_payload['edge_weight'].float()


def build_or_load_correlation_graph(returns_df, tickers, k_edges):
    if CORR_GRAPH_PATH.exists():
        return torch.load(CORR_GRAPH_PATH, map_location='cpu')
    train_returns = returns_df.loc[:TRAIN_END]
    corr = train_returns.corr().abs().fillna(0.0)
    candidates = []
    for i, a in enumerate(tickers):
        for j, b in enumerate(tickers):
            if i != j:
                candidates.append((float(corr.loc[a, b]), i, j, a, b))
    candidates.sort(reverse=True, key=lambda x: x[0])
    selected = candidates[:max(1, int(k_edges))]
    edge_index = torch.tensor([[i for _, i, _, _, _ in selected], [j for _, _, j, _, _ in selected]], dtype=torch.long)
    edge_weight = torch.tensor([score for score, *_ in selected], dtype=torch.float32)
    payload = {
        'ticker_list': tickers,
        'edge_index': edge_index,
        'edge_weight': edge_weight,
        'method': 'top_abs_correlation_density_matched',
        'n_edges': int(edge_index.shape[1]),
        'matched_to_causal_edges': int(k_edges),
    }
    torch.save(payload, CORR_GRAPH_PATH)
    log_event('correlation_graph_saved', path=str(CORR_GRAPH_PATH), n_edges=int(edge_index.shape[1]))
    return payload


causal_edge_index, causal_edge_weight = load_graph_tensors(causal_graph)
if causal_edge_index.shape[1] == 0:
    print('WARNING: causal graph is empty after FDR; inspect FDR_ALPHA before interpreting GNN results.')
correlation_graph = build_or_load_correlation_graph(returns_df, tickers, k_edges=max(1, causal_edge_index.shape[1]))
corr_edge_index, corr_edge_weight = load_graph_tensors(correlation_graph)
no_edge_index, no_edge_weight = empty_graph(len(tickers))

print(f"Causal graph edges: {causal_edge_index.shape[1]:,}")
print(f"Correlation graph edges: {corr_edge_index.shape[1]:,}")
print('GRU-only baseline uses no graph propagation (use_gcn=False).')

## Phase B4 - Resumable Training And Evaluation

Each model checkpoints to Drive every `CHECKPOINT_EVERY_EPOCHS`, and writes an emergency checkpoint on exceptions. Rerunning this cell resumes from the newest checkpoint.

In [ ]:
def latest_checkpoint(model_name):
    paths = sorted((CHECKPOINT_DIR / model_name).glob('epoch_*.pt'))
    return paths[-1] if paths else None


def save_checkpoint(model_name, epoch, model, optimizer, best_val_f1, extra=None):
    out_dir = CHECKPOINT_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f'epoch_{epoch:04d}.pt'
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_f1': best_val_f1,
        'extra': extra or {},
    }, path)
    return path


@torch.no_grad()
def evaluate(model, loader, edge_index, edge_weight, loss_fn):
    model.eval()
    losses, all_pred, all_true = [], [], []
    edge_index = edge_index.to(DEVICE)
    edge_weight = edge_weight.to(DEVICE)
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x, edge_index, edge_weight)
        losses.append(float(loss_fn(logits, y).item()))
        all_pred.append((torch.sigmoid(logits) >= 0.5).detach().cpu().numpy().astype(int).ravel())
        all_true.append(y.detach().cpu().numpy().astype(int).ravel())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    return {
        'loss': float(np.mean(losses)) if losses else float('nan'),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'positive_rate_true': float(y_true.mean()),
        'positive_rate_pred': float(y_pred.mean()),
    }


def train_model(model_name, edge_index, edge_weight, use_gcn=True, epochs=EPOCHS):
    print(f"\n=== Training {model_name} | use_gcn={use_gcn} | device={DEVICE} ===")
    model = CausalTemporalGNN(hidden=HIDDEN, use_gcn=use_gcn).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.BCEWithLogitsLoss()
    start_epoch = 1
    best_val_f1 = -1.0
    ckpt = latest_checkpoint(model_name)
    if ckpt:
        payload = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(payload['model_state'])
        optimizer.load_state_dict(payload['optimizer_state'])
        start_epoch = int(payload['epoch']) + 1
        best_val_f1 = float(payload.get('best_val_f1', -1.0))
        print(f"Resuming from {ckpt} at epoch {start_epoch}")

    edge_index = edge_index.to(DEVICE)
    edge_weight = edge_weight.to(DEVICE)
    history = []
    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()
            train_losses = []
            pbar = tqdm(train_loader, desc=f'{model_name} epoch {epoch}/{epochs}')
            for x, y in pbar:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x, edge_index, edge_weight)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                train_losses.append(float(loss.item()))
                pbar.set_postfix(loss=f"{np.mean(train_losses):.4f}")

            val_metrics = evaluate(model, val_loader, edge_index, edge_weight, loss_fn)
            row = {'epoch': epoch, 'train_loss': float(np.mean(train_losses)), **{f'val_{k}': v for k, v in val_metrics.items()}}
            history.append(row)
            print(row)
            log_event('epoch_complete', model=model_name, **row)

            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                best_path = save_checkpoint(model_name, epoch, model, optimizer, best_val_f1, extra={'best': True, 'history': history})
                shutil.copy2(best_path, CHECKPOINT_DIR / model_name / 'best.pt')
                print(f"New best validation F1: {best_val_f1:.4f}")

            if epoch % CHECKPOINT_EVERY_EPOCHS == 0:
                print(f"Checkpoint saved: {save_checkpoint(model_name, epoch, model, optimizer, best_val_f1, extra={'history': history})}")

        best_file = CHECKPOINT_DIR / model_name / 'best.pt'
        if best_file.exists():
            payload = torch.load(best_file, map_location=DEVICE)
            model.load_state_dict(payload['model_state'])
        test_metrics = evaluate(model, test_loader, edge_index, edge_weight, loss_fn)
        log_event('model_test_complete', model=model_name, **test_metrics)
        save_checkpoint(model_name, epochs, model, optimizer, best_val_f1, extra={'test_metrics': test_metrics, 'final': True})
        return {'model': model_name, 'best_val_f1': best_val_f1, 'test': test_metrics}
    except BaseException as exc:
        emergency = save_checkpoint(model_name, max(start_epoch - 1, 0), model, optimizer, best_val_f1, extra={'emergency_error': str(exc)})
        print(f"Emergency checkpoint saved after exception: {emergency}")
        log_event('training_exception_checkpoint_saved', model=model_name, checkpoint=str(emergency), error=str(exc))
        raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# Train the primary model and both baselines.
results = []
results.append(train_model('causal_granger_fdr_gnn', causal_edge_index, causal_edge_weight, use_gcn=True))
results.append(train_model('baseline_gru_only_no_graph', no_edge_index, no_edge_weight, use_gcn=False))
results.append(train_model('baseline_correlation_graph_gnn', corr_edge_index, corr_edge_weight, use_gcn=True))

summary = {
    'device': str(DEVICE),
    'n_stocks': len(tickers),
    'window': WINDOW,
    'split': {
        'train_end': str(TRAIN_END.date()),
        'val': [str(VAL_START.date()), str(VAL_END.date())],
        'test': [str(TEST_START.date()), str(TEST_END.date())],
    },
    'causal_graph_edges': int(causal_edge_index.shape[1]),
    'correlation_graph_edges': int(corr_edge_index.shape[1]),
    'results': results,
}
TRAINING_SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print(json.dumps(summary, indent=2, default=str))
print(f"Training summary saved to Drive: {TRAINING_SUMMARY_PATH}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def get_probs(model, X, edge_index=None, edge_weight=None):
    model.eval()
    with torch.no_grad():
        logits = torch.stack([model(X[s].to(device), edge_index, edge_weight)
                                for s in range(X.shape[0])])
    return torch.sigmoid(logits).cpu().numpy().flatten()

def sweep_thresholds(model, edge_index=None, edge_weight=None):
    val_probs = get_probs(model, X_val, edge_index, edge_weight)
    val_actual = y_val.numpy().flatten()

    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.30, 0.70, 0.02):
        preds = (val_probs > t).astype(float)
        f1 = f1_score(val_actual, preds, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    # now apply that chosen threshold to the TEST set (never tune on test data)
    test_probs = get_probs(model, X_test, edge_index, edge_weight)
    test_actual = y_test.numpy().flatten()
    test_preds = (test_probs > best_t).astype(float)

    return {
        "best_threshold": round(best_t, 2),
        "test_precision": round(precision_score(test_actual, test_preds, zero_division=0), 4),
        "test_recall": round(recall_score(test_actual, test_preds, zero_division=0), 4),
        "test_f1": round(f1_score(test_actual, test_preds, zero_division=0), 4),
        "predicted_positive_rate": round(test_preds.mean(), 4),  # THE key sanity check
    }

print("Causal graph:      ", sweep_thresholds(model_causal, edge_index, edge_weight))
print("No graph:          ", sweep_thresholds(model_nograph))
print("Correlation graph: ", sweep_thresholds(model_corr, corr_edge_index, corr_edge_weight))

## Final Reporting Checklist

Drive artifacts to report:

- `graphs/nifty500_granger_fdr_graph.pt`
- `graphs/nifty500_granger_fdr_edges.csv`
- `graphs/nifty500_louvain_clusters.json`
- `checkpoints/<model_name>/`
- `logs/training_summary.json`

Guardrails:

- Do not claim PCMCI was run on all 500 variables unless you explicitly run the 50-stock timing gate and then a justified full or sector-restricted pass.
- Do not compare against random splits; metrics use the chronological 2019/2020/2021 walk-forward split.
- If BH-FDR produces a sparse graph, report that result honestly rather than relaxing alpha silently.